<a href="https://colab.research.google.com/github/jackevansadl/chem2002/blob/main/C2/02_MO_Diagrams_N2_O2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: Molecular Orbital Diagrams of N$_2$ and O$_2$ ⚛️

In notebook 1 you combined two atomic orbitals into a bonding and an antibonding molecular orbital. For molecules with more electrons we stack *all* the molecular orbitals into an **MO energy-level diagram** and fill it with electrons, just like an Aufbau diagram for atoms.

This notebook tackles a famous puzzle. The simple Lewis structure of **oxygen**, O=O, shows all electrons paired. Yet liquid oxygen is **paramagnetic** — it sticks to a magnet — which means it has *unpaired* electrons. Lewis theory gets this **wrong**; MO theory gets it **right**. Let's see how.

We will:
* build MO diagrams for N$_2$ and O$_2$ and read off their **bond order**,
* visualise the real $\sigma$ and $\pi$ orbital shapes with PySCF,
* **calculate** the number of unpaired electrons in O$_2$ and confirm it is paramagnetic.

In [ ]:
!pip install -q pyscf py3Dmol matplotlib

In [ ]:
# Imports and helper functions
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol
from pyscf import gto, scf
from pyscf.tools import cubegen

BOHR = 0.529177

def mol_to_xyz(mol):
    coords = mol.atom_coords() * BOHR
    lines = [str(mol.natm), ""]
    for i in range(mol.natm):
        x, y, z = coords[i]
        lines.append(f"{mol.atom_symbol(i)} {x:.4f} {y:.4f} {z:.4f}")
    return chr(10).join(lines)

def show_orbital(mol, coeff, isoval=0.03):
    cubegen.orbital(mol, "/tmp/orb.cube", coeff)
    cube = open("/tmp/orb.cube").read()
    v = py3Dmol.view(width=420, height=380)
    v.addModel(mol_to_xyz(mol), "xyz")
    v.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
    v.addVolumetricData(cube, "cube", {"isoval":  isoval, "color": "blue", "opacity": 0.85})
    v.addVolumetricData(cube, "cube", {"isoval": -isoval, "color": "red",  "opacity": 0.85})
    v.zoomTo()
    return v.show()

# ---- Molecular-orbital energy-level diagrams ----
# Each level is (label, degeneracy, character) where character "b" = bonding, "a" = antibonding.
# There are two standard orderings for second-row diatomics:
ORDER_EARLY = [("σ2s",1,"b"),("σ*2s",1,"a"),("π2p",2,"b"),("σ2p",1,"b"),("π*2p",2,"a"),("σ*2p",1,"a")]  # Li2..N2
ORDER_LATE  = [("σ2s",1,"b"),("σ*2s",1,"a"),("σ2p",1,"b"),("π2p",2,"b"),("π*2p",2,"a"),("σ*2p",1,"a")]   # O2, F2

def fill_levels(order, n_val):
    """Fill n_val valence electrons (Hund's rule) -> list of electron counts per level."""
    e = n_val; occ = []
    for label, deg, kind in order:
        ne = min(e, 2 * deg); e -= ne; occ.append(ne)
    return occ

def bond_order_and_spin(order, n_val):
    occ = fill_levels(order, n_val); b = a = unp = 0
    for (label, deg, kind), ne in zip(order, occ):
        comp = [0] * deg
        for _ in range(ne):                       # Hund: singly fill, then pair
            comp[comp.index(min(comp))] += 1
        unp += sum(1 for c in comp if c == 1)
        if kind == "b": b += ne
        else: a += ne
    return (b - a) / 2, unp

def draw_mo_diagram(order, n_val, title=""):
    occ = fill_levels(order, n_val)
    fig, ax = plt.subplots(figsize=(4.5, 6))
    for y, ((label, deg, kind), ne) in enumerate(zip(order, occ)):
        color = "#1f77b4" if kind == "b" else "#d62728"
        xs = [0.0] if deg == 1 else [-0.5, 0.5]
        comp = [0] * deg
        for _ in range(ne):
            comp[comp.index(min(comp))] += 1
        for xc, c in zip(xs, comp):
            ax.hlines(y, xc - 0.3, xc + 0.3, color=color, lw=2.5)
            if c >= 1:
                ax.annotate("", xy=(xc - 0.08, y + 0.28), xytext=(xc - 0.08, y - 0.28),
                            arrowprops=dict(arrowstyle="->", color="k", lw=1.3))
            if c == 2:
                ax.annotate("", xy=(xc + 0.08, y - 0.28), xytext=(xc + 0.08, y + 0.28),
                            arrowprops=dict(arrowstyle="->", color="k", lw=1.3))
        ax.text(1.15, y, label, va="center", fontsize=12)
    bo, unp = bond_order_and_spin(order, n_val)
    ax.set_xlim(-1.6, 2.3); ax.set_ylim(-0.8, len(order)-0.2)
    ax.axis("off"); ax.set_title(title, fontsize=13)
    ax.text(-1.5, len(order)-1, "antibonding", color="#d62728", fontsize=9)
    ax.text(-1.5, 0, "bonding", color="#1f77b4", fontsize=9)
    plt.show()
    print(f"Bond order = {bo:.1f}    Unpaired electrons = {unp}")

-----
## 2.1 Bond Order

The **bond order** counts the net number of bonds:

$$\text{bond order} = \tfrac{1}{2}\,(\text{bonding electrons} - \text{antibonding electrons})$$

A higher bond order means a **stronger, shorter** bond. In the diagrams below, **blue** levels are bonding and **red** levels are antibonding. We only need the **valence** electrons (the 1s core electrons cancel out).

## 2.2 Nitrogen, N$_2$

Nitrogen has 5 valence electrons per atom = **10 valence electrons**. Run the cell to build and fill its MO diagram.

In [ ]:
# N2: 10 valence electrons, "early" ordering (pi below sigma)
draw_mo_diagram(ORDER_EARLY, n_val=10, title="MO diagram of N$_2$")

A bond order of **3** — the famous **triple bond** of N$_2$, which makes it extremely stable and unreactive. All electrons are paired, so N$_2$ is **diamagnetic** (not magnetic).

## 2.3 Seeing $\sigma$ and $\pi$ Orbitals

The diagram labels orbitals $\sigma$ or $\pi$. What do these actually look like?

* A **$\sigma$** orbital is cylindrically symmetric around the bond axis.
* A **$\pi$** orbital has a **nodal plane** that contains the bond axis (density above and below, like two sausages).

Let's compute N$_2$ and view its highest occupied $\pi$ orbital and the empty $\pi^*$ orbital.

In [ ]:
n2 = gto.M(atom="N 0 0 0; N 0 0 1.10", basis="6-31g*", verbose=0)
mf = scf.RHF(n2).run()
n_occ = n2.nelectron // 2
print(f"HOMO energy: {mf.mo_energy[n_occ-1]*27.211:.2f} eV   LUMO energy: {mf.mo_energy[n_occ]*27.211:.2f} eV")

# Occupied pi orbital (HOMO region) -- note the nodal plane along the bond
show_orbital(n2, mf.mo_coeff[:, n_occ - 1])

In [ ]:
# The empty antibonding pi* orbital (LUMO)
show_orbital(n2, mf.mo_coeff[:, n_occ])

## 2.4 Oxygen, O$_2$ — the Paramagnetism Puzzle

Oxygen has 6 valence electrons per atom = **12 valence electrons**. Two more than N$_2$, and those two extra electrons go into the **doubly-degenerate $\pi^*$** level. By **Hund's rule** they occupy the two $\pi^*$ orbitals *singly, with parallel spins* — giving **two unpaired electrons**.

In [ ]:
# O2: 12 valence electrons, "late" ordering (sigma below pi)
draw_mo_diagram(ORDER_LATE, n_val=12, title="MO diagram of O$_2$")

In [ ]:
# Confirm it with a calculation: O2 is computed as a spin-triplet (2 unpaired e-)
o2 = gto.M(atom="O 0 0 0; O 0 0 1.21", basis="6-31g*", spin=2, verbose=0)
mf_o2 = scf.UHF(o2).run()
n_alpha = int(round((mf_o2.mo_occ[0] > 0).sum()))
n_beta  = int(round((mf_o2.mo_occ[1] > 0).sum()))
print(f"Unpaired electrons in O2 = {n_alpha - n_beta}")
print("=> O2 has unpaired electrons, so it is PARAMAGNETIC (Lewis structure cannot explain this).")

-----
### **📋 Reporting Task 2**

Using the `draw_mo_diagram` function and the bond-order formula, complete the table in your workbook for the following species. For each, report the **bond order** and whether it is **paramagnetic** (has unpaired electrons) or **diamagnetic**.

| Species | Valence electrons | Ordering | Bond order | Para/diamagnetic |
|---|---|---|---|---|
| N$_2$ | 10 | early | ? | ? |
| O$_2$ | 12 | late | ? | ? |
| O$_2^{+}$ (remove 1 e$^-$) | 11 | late | ? | ? |
| F$_2$ | 14 | late | ? | ? |

Then answer: **how does the bond order of O$_2^{+}$ compare to O$_2$, and what does that predict about its bond length?**

In [ ]:
# Use draw_mo_diagram(order, n_val) to fill in your table.
# Example for O2+ :
# draw_mo_diagram(ORDER_LATE, n_val=11, title="O2+")

-----
## Next Steps

You've used MO theory to explain bonding, bond order, and the magnetism of O$_2$ — something Lewis structures cannot do.

➡️ Continue to **`03_Crystal_Field_TM_Complexes.ipynb`**, where we apply these ideas to **transition-metal complexes**: how the *d* orbitals split in a metal complex, and why that gives these compounds their striking **colours** and magnetic behaviour.